## Statistics as a DataFrame with `summarize()`

`plot(df)` and `create_report(df)` produce HTML for people to read. `summarize(df)` returns the same statistics and data-quality insights as a pandas DataFrame, with one row per column, so you can filter them, save them, compare them between runs or check them in a test.

The columns of the summary are always the same (`dataprep.eda.summary.SUMMARY_COLUMNS`); a statistic that does not apply to a column, such as the mean of a text column, is `NaN`.

In [ ]:
from dataprep.datasets import load_dataset
from dataprep.eda import summarize

df = load_dataset("titanic")
summary = summarize(df)
summary

Select what you need with ordinary pandas code. Percentages are out of 100, and `insights` lists the findings of `plot(df)` for each column, such as `Missing` or `Skewed`:

In [ ]:
summary.loc[summary["missing_pct"] > 5, ["missing", "missing_pct"]]

In [ ]:
summary[summary["insights"].str.contains("Skewed")].index.tolist()

Table-level figures (rows, duplicate rows, missing cells, memory) are in `summary.attrs["dataset"]`:

In [ ]:
summary.attrs["dataset"]

A summary makes a simple data-quality check in a pipeline or a test:

```python
summary = summarize(new_batch)
assert summary.loc["customer_id", "missing"] == 0
assert summary.loc["age", "max"] < 120
assert summary.attrs["dataset"]["duplicate_rows"] == 0
```

The insight thresholds take the same `config` keys as `plot`, e.g. `summarize(df, config={"insight.missing.threshold": 10})`.

## Which columns relate to a target?

`plot(df, target="...")` ranks every column by how strongly it is associated with a target column, and shows the distribution of the target. It is a quick way to see which columns could help predict the target, and to catch columns that leak it.

In [ ]:
from dataprep.eda import plot

plot(df, target="Survived")

Each column gets a score from 0 (no association) to 1 (the column determines the target). The measure depends on the types of the two columns:

| column | target | score |
|---|---|---|
| numerical | numerical | absolute Spearman rank correlation |
| numerical | categorical | correlation ratio (eta) |
| categorical | numerical | correlation ratio (eta) |
| categorical | categorical | Cramér's V |

Numbers with fewer than 10 distinct values, such as 0/1 labels, count as categorical. The correlation ratio and Cramér's V are bias-corrected, so a column with a different value in nearly every row (such as an ID) does not look important just because it splits the rows finely. The **!** button lists warnings: columns that look like identifiers, scores so high that the column probably leaks the target, and imbalanced classes.

`rank_features()` returns the ranking as a DataFrame:

In [ ]:
from dataprep.eda import rank_features

rank_features(df, "Survived")

`summarize(df, target="Survived")` adds the score and the method to the summary, and `plot(df, "Age", target="Survived")` is the same as `plot(df, "Age", "Survived")`.

To treat a numerical target as classes, override its type: `rank_features(df, "rating", dtype={"rating": "nominal"})`.

## Large tables: the `sample` option

Every EDA function takes `sample=`: a number of rows (`sample=100_000`) or a fraction of the rows (`sample=0.1`). The function then works on a random sample of the rows, which makes it much faster on large tables. The sample is the same on every run, the output says that it shows a sample, and asking for at least as many rows as the table has uses every row.

Counts in the output (missing values, duplicates, distinct values) are counts in the sample, not in the full table. Rare values and rare problems can be missed by a sample, so profile the full table before relying on the absence of something.

In [ ]:
from dataprep.eda import create_report, plot_correlation

plot(df, sample=500)

In [ ]:
plot_correlation(load_dataset("house_prices_train"), sample=0.5)

`sample` works the same way for `plot_missing`, `create_report`, `summarize` and `rank_features`. For `plot_diff` and `create_diff_report` it samples each DataFrame. Dask DataFrames are sampled before they are loaded into memory; their sample size is approximate.